# PaliGemma 2 QLoRA Fine-Tuning for Military Object Detection

Fine-tunes PaliGemma 2 10B (896px) on a custom military dataset using QLoRA.
Trains LoRA adapters on the Gemma 2 decoder while keeping the SigLIP vision encoder
and multimodal projector frozen.

**Detection classes:** Tank, AirCraft

**Dataset:** Roboflow `sanders-workspace/military-f5tbj-bfk2p` v4 (paligemma format)

## 1. Setup

In [ ]:
!pip install -U supervision
!pip install -q transformers accelerate peft bitsandbytes datasets pillow roboflow
!pip install -q pillow matplotlib numpy opencv-python requests

## 2. Model Loading & QLoRA Configuration

In [ ]:
import torch
import numpy as np
from PIL import Image
import supervision as sv
from transformers import BitsAndBytesConfig

from transformers import (
    PaliGemmaProcessor,
    PaliGemmaForConditionalGeneration,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from google.colab import drive
drive.mount("/content/drive")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.bfloat16

MODEL_ID = "google/paligemma2-10b-pt-896"
MODEL_DIR = "/content/drive/MyDrive/paligemma_tank_model-ferdig-test"

CLASSES = ["Tank", "AirCraft"]

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

processor = PaliGemmaProcessor.from_pretrained(MODEL_ID)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# prepare for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# apply LoRA
model = get_peft_model(model, lora_config)

# freeze vision encoder and multimodal projector
model.base_model.model.model.vision_tower.requires_grad_(False)
model.base_model.model.model.multi_modal_projector.requires_grad_(False)

model.print_trainable_parameters()

## 3. Dataset

In [ ]:
import os
import json
import random
from PIL import Image
from torch.utils.data import Dataset

from google.colab import userdata
from roboflow import Roboflow

#DRIVE
from google.colab import drive
drive.mount('/content/drive')

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)


project = rf.workspace("sanders-workspace").project("military-f5tbj-bfk2p")
version = project.version(4)
dataset = version.download("paligemma")

class JSONLDataset(Dataset):
    def __init__(self, jsonl_file_path: str, image_directory_path: str):
        self.jsonl_file_path = jsonl_file_path
        self.image_directory_path = image_directory_path
        self.entries = self._load_entries()

    def _load_entries(self):
        entries = []
        with open(self.jsonl_file_path, 'r') as file:
            for line in file:
                data = json.loads(line)
                entries.append(data)
        return entries

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx: int):
        if idx < 0 or idx >= len(self.entries):
            raise IndexError("Index out of range")

        entry = self.entries[idx]
        image_path = os.path.join(self.image_directory_path, entry['image'])
        image = Image.open(image_path)
        return image, entry

train_dataset = JSONLDataset(
    jsonl_file_path=f"{dataset.location}/dataset/_annotations.train.jsonl",
    image_directory_path=f"{dataset.location}/dataset",
)
valid_dataset = JSONLDataset(
    jsonl_file_path=f"{dataset.location}/dataset/_annotations.valid.jsonl",
    image_directory_path=f"{dataset.location}/dataset",
)
test_dataset = JSONLDataset(
    jsonl_file_path=f"{dataset.location}/dataset/_annotations.test.jsonl",
    image_directory_path=f"{dataset.location}/dataset",
)

## 4. Data Collation

In [ ]:
def collate_fn(batch):

    images = []
    prefixes = []
    suffixes = []

    for image, label in batch:

        images.append(image)

        prefixes.append("<image>" + label["prefix"])

        suffix = label["suffix"].strip()

        # append stop token
        if not suffix.endswith("<end>"):
            suffix = suffix + "\n<end>"

        suffixes.append(suffix)

    inputs = processor(
        text=prefixes,
        images=images,
        suffix=suffixes,
        return_tensors="pt",
        padding="longest"
    )

    inputs = {
        k: v.to(TORCH_DTYPE) if v.dtype == torch.float else v
        for k, v in inputs.items()
    }

    return inputs

## 5. Training

In [ ]:
training_args = TrainingArguments(

    output_dir=MODEL_DIR,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    max_steps=3000,

    logging_steps=10,
    save_steps=200,

    bf16=True,
    eval_strategy="steps",
    eval_steps=100,
    per_device_eval_batch_size=1,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,

    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collate_fn
)

trainer.train()
trainer.model.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)

## 6. Load Model from Checkpoint

Reload the base model with the trained LoRA adapters for inference and evaluation.

In [ ]:
import torch
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "google/paligemma2-10b-pt-896"
# path to the directory containing the adapter weights (safetensors)
FINETUNED_MODEL_DIR = "/content/drive/MyDrive/paligemma_tank_model-ferdig-test/checkpoint-200"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading base model...")
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"
)

print(f"Loading LoRA adapters from {FINETUNED_MODEL_DIR}...")
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_DIR)
model.eval()

processor = PaliGemmaProcessor.from_pretrained(BASE_MODEL_ID)

print("Model loaded and ready for inference.")

## 7. Inference — Detection + Attention Heatmaps

Runs detection on test images, draws bounding boxes, and extracts per-object
attention heatmaps from the Gemma 2 decoder. Includes NMS to remove duplicate detections.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import gc
import re

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union_area = box1_area + box2_area - inter_area
    if union_area == 0: return 0
    return inter_area / union_area

def process_image_and_get_heatmaps_896(img_idx):
    # 1. Load and resize image
    raw_image, label = test_dataset[img_idx]
    prompt = "<image>\n" + label["prefix"]

    image_resized = raw_image.resize((896, 896))
    inputs = processor(text=prompt, images=image_resized, return_tensors="pt").to(DEVICE)

    # 2. Inference with attention output
    print(f"\n--- Image {img_idx + 1} ---")
    print("Running inference and extracting attention...")

    model.eval()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            output_attentions=True,
            return_dict_in_generate=True
        )

    # 3. Map generated tokens to detected objects
    input_len = inputs["input_ids"].shape[1]
    gen_tokens = outputs.sequences[0][input_len:]

    raw_objects = []
    current_box_tokens = []
    current_label_steps = []

    for step, token_id in enumerate(gen_tokens):
        token_str = processor.decode(token_id, skip_special_tokens=False)

        if ";" in token_str or "<end>" in token_str or "<eos>" in token_str or token_id == processor.tokenizer.eos_token_id:
            if len(current_box_tokens) == 4 and len(current_label_steps) > 0:
                raw_objects.append({
                    "loc_tokens": current_box_tokens,
                    "label_steps": current_label_steps
                })
            current_box_tokens = []
            current_label_steps = []
            continue

        if "<loc" in token_str:
            current_box_tokens.append(token_str)
        else:
            if len(current_box_tokens) == 4:
                current_label_steps.append(step)

    if len(current_box_tokens) == 4 and len(current_label_steps) > 0:
        raw_objects.append({
            "loc_tokens": current_box_tokens,
            "label_steps": current_label_steps
        })

    # 3.5 NMS filtering
    objects = []
    iou_threshold = 0.3

    for obj in raw_objects:
        coords = [int(re.search(r'\d+', loc).group()) for loc in obj["loc_tokens"]]
        if len(coords) != 4: continue

        y_min, x_min, y_max, x_max = coords
        box = [
            int((x_min / 1024.0) * 896),
            int((y_min / 1024.0) * 896),
            int((x_max / 1024.0) * 896),
            int((y_max / 1024.0) * 896)
        ]

        obj["box"] = box

        keep = True
        for kept_obj in objects:
            if compute_iou(box, kept_obj["box"]) > iou_threshold:
                keep = False
                break

        if keep:
            objects.append(obj)

    print(f"Found {len(raw_objects)} raw objects. NMS filtered to {len(objects)} unique detections.")

    # 4. Locate image tokens
    image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")
    img_mask = (inputs["input_ids"][0] == image_token_id).cpu().numpy()
    img_indices = np.where(img_mask)[0]

    if len(img_indices) == 4096:
        img_start = img_indices[0]
        img_end = img_indices[-1] + 1
    else:
        img_start = 1 if inputs["input_ids"][0][0].item() == processor.tokenizer.bos_token_id else 0
        img_end = img_start + 4096

    # 5. Draw bounding boxes and extract per-object heatmaps
    visualizations = []
    img_with_boxes = np.array(image_resized).copy()

    for obj in objects:
        box = obj["box"]
        label_text = processor.decode([gen_tokens[i] for i in obj["label_steps"]], skip_special_tokens=True).strip()

        cv2.rectangle(img_with_boxes, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
        text_y = max(20, box[1] - 8)
        cv2.putText(img_with_boxes, label_text, (box[0], text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        token_attns = []
        for step in obj["label_steps"]:
            step_layers = list(outputs.attentions[step])
            if isinstance(step_layers[0], tuple):
                step_layers = [layer[0] for layer in step_layers]

            max_seq_len = max(layer.shape[-1] for layer in step_layers)

            valid_layers = []
            for layer in step_layers:
                if layer.shape[-1] == max_seq_len:
                    img_attn = layer[0, :, 0, img_start:img_end].mean(dim=0)
                    valid_layers.append(img_attn)

            if len(valid_layers) > 0:
                avg_valid_layers = torch.stack(valid_layers).mean(dim=0)
                token_attns.append(avg_valid_layers)

        if len(token_attns) == 0:
            continue

        final_attn = torch.stack(token_attns).mean(dim=0).to(torch.float32).cpu().numpy()

        heatmap_grid = final_attn.reshape(64, 64)

        # percentile normalization
        v_min = heatmap_grid.min()
        v_max = np.percentile(heatmap_grid, 99.5)
        heatmap_norm = np.clip((heatmap_grid - v_min) / (v_max - v_min + 1e-8), 0, 1)

        heatmap_resized_cv = cv2.resize(heatmap_norm, (896, 896), interpolation=cv2.INTER_CUBIC)

        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized_cv), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

        overlay = cv2.addWeighted(np.array(image_resized), 0.5, heatmap_colored, 0.5, 0)
        visualizations.append((label_text, overlay))

    # 6. Plot results
    if len(objects) > 0:
        cols = min(4, 1 + len(visualizations))
        rows = int(np.ceil((1 + len(visualizations)) / cols))

        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
        axes = np.array(axes).flatten()

        axes[0].imshow(img_with_boxes)
        axes[0].set_title(f"Pass 1: {len(objects)} Unique Detections")
        axes[0].axis('off')

        for i, (label_txt, overlay_img) in enumerate(visualizations):
            axes[i+1].imshow(overlay_img)
            axes[i+1].set_title(f"Attention: '{label_txt}'")
            axes[i+1].axis('off')

        for j in range(1 + len(visualizations), len(axes)):
            axes[j].axis('off')

        plt.tight_layout()
        plt.show()

    # 7. VRAM cleanup
    del inputs, outputs, gen_tokens, raw_objects, objects, img_with_boxes, visualizations
    if 'token_attns' in locals():
        del token_attns
    if 'valid_layers' in locals():
        del valid_layers, avg_valid_layers, final_attn, heatmap_grid, overlay

    clear_vram()

# run on first 20 test images
for i in range(20):
    process_image_and_get_heatmaps_896(img_idx=i)

## 8. Evaluation — Precision / Recall / F1

In [ ]:
import re
import numpy as np
import torch
import gc
from collections import defaultdict

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compute_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def parse_paligemma_string(text):
    """Parse PaliGemma output string into a list of boxes and labels."""
    objects = []
    parts = text.split(";")
    for part in parts:
        part = part.strip()
        if not part: continue

        locs = re.findall(r'<loc(\d{4})>', part)
        if len(locs) >= 4:
            y_min, x_min, y_max, x_max = [int(l) for l in locs[:4]]

            box = [
                int((x_min / 1024.0) * 896), int((y_min / 1024.0) * 896),
                int((x_max / 1024.0) * 896), int((y_max / 1024.0) * 896)
            ]

            label = re.sub(r'<loc\d{4}>', '', part).replace('<end>', '').replace('<eos>', '').strip()
            objects.append({"box": box, "label": label})
    return objects

def evaluate_model(dataset, num_images_to_test=None):
    if num_images_to_test is None:
        num_images_to_test = len(dataset)

    print(f"Evaluating {num_images_to_test} images...\n")

    stats = defaultdict(lambda: {"TP": 0, "FP": 0, "FN": 0})

    model.eval()

    for i in range(num_images_to_test):
        image, label_data = dataset[i]

        # ground truth
        gt_text = label_data["suffix"]
        gt_objects = parse_paligemma_string(gt_text)

        # inference
        prompt = "<image>\n" + label_data["prefix"]
        image_resized = image.resize((896, 896))
        inputs = processor(text=prompt, images=image_resized, return_tensors="pt").to(DEVICE)

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)

        input_len = inputs["input_ids"].shape[1]
        gen_tokens = outputs[0][input_len:]
        pred_text = processor.decode(gen_tokens, skip_special_tokens=False)

        # parse predictions + NMS
        raw_preds = parse_paligemma_string(pred_text)
        pred_objects = []
        for p_obj in raw_preds:
            keep = True
            for kept_obj in pred_objects:
                if p_obj["label"] == kept_obj["label"] and compute_iou(p_obj["box"], kept_obj["box"]) > 0.2:
                    keep = False
                    break
            if keep:
                pred_objects.append(p_obj)

        # IoU matching per class
        all_classes = set([obj["label"] for obj in gt_objects] + [obj["label"] for obj in pred_objects])

        for cls in all_classes:
            gt_boxes = [obj["box"] for obj in gt_objects if obj["label"] == cls]
            pr_boxes = [obj["box"] for obj in pred_objects if obj["label"] == cls]

            matched_gt = set()

            for p_box in pr_boxes:
                best_iou = 0
                best_gt_idx = -1

                for idx, g_box in enumerate(gt_boxes):
                    if idx in matched_gt: continue
                    iou = compute_iou(p_box, g_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = idx

                if best_iou >= 0.5:
                    stats[cls]["TP"] += 1
                    matched_gt.add(best_gt_idx)
                else:
                    stats[cls]["FP"] += 1

            stats[cls]["FN"] += len(gt_boxes) - len(matched_gt)

        clear_vram()
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1} / {num_images_to_test} images...")

    # results
    print("\n" + "="*50)
    print(" EVALUATION REPORT (IoU > 0.50)")
    print("="*50)

    for cls, s in stats.items():
        tp, fp, fn = s["TP"], s["FP"], s["FN"]

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"\nClass: **{cls}**")
        print(f"  - True Positives (TP): {tp}")
        print(f"  - False Positives (FP): {fp}")
        print(f"  - False Negatives (FN): {fn}")
        print(f"  --> Precision: {precision:.3f}")
        print(f"  --> Recall:    {recall:.3f}")
        print(f"  --> F1-Score:  {f1:.3f}")

evaluate_model(test_dataset)

## 9. Evaluation — Confusion Matrix

In [ ]:
import re
import numpy as np
import torch
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

IMAGE_SIZE = 896
NMS_THRESHOLD = 0.15
IOU_MATCH_THRESHOLD = 0.50
CLASSES = ["AirCraft", "Tank", "Background"]

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def compute_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def parse_paligemma_string(text, img_size):
    objects = []
    for part in text.split(";"):
        part = part.strip()
        if not part: continue
        locs = re.findall(r'<loc(\d{4})>', part)
        if len(locs) >= 4:
            y_min, x_min, y_max, x_max = [int(l) for l in locs[:4]]
            box = [
                int((x_min / 1024.0) * img_size), int((y_min / 1024.0) * img_size),
                int((x_max / 1024.0) * img_size), int((y_max / 1024.0) * img_size)
            ]
            label = re.sub(r'<loc\d{4}>', '', part).replace('<end>', '').replace('<eos>', '').strip()

            # normalize label casing
            if label.lower() == "aircraft": label = "AirCraft"
            if label.lower() == "tank": label = "Tank"

            objects.append({"box": box, "label": label})
    return objects

def generate_confusion_matrix(dataset):
    print(f"Building confusion matrix for {len(dataset)} images...\n")
    model.eval()

    conf_matrix = {true_cls: {pred_cls: 0 for pred_cls in CLASSES} for true_cls in CLASSES}

    for i in range(len(dataset)):
        image, label_data = dataset[i]

        gt_objects = parse_paligemma_string(label_data["suffix"], IMAGE_SIZE)

        prompt = "<image>\n" + label_data["prefix"]
        image_resized = image.resize((IMAGE_SIZE, IMAGE_SIZE))
        inputs = processor(text=prompt, images=image_resized, return_tensors="pt").to(DEVICE)

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)

        gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        pred_text = processor.decode(gen_tokens, skip_special_tokens=False)

        # NMS
        raw_preds = parse_paligemma_string(pred_text, IMAGE_SIZE)
        pred_objects = []
        for p_obj in raw_preds:
            keep = True
            for kept_obj in pred_objects:
                if p_obj["label"] == kept_obj["label"] and compute_iou(p_obj["box"], kept_obj["box"]) > NMS_THRESHOLD:
                    keep = False
                    break
            if keep: pred_objects.append(p_obj)

        # matching
        matched_gt_indices = set()

        for p_idx, p_obj in enumerate(pred_objects):
            best_iou = 0
            best_gt_idx = -1

            for g_idx, g_obj in enumerate(gt_objects):
                if g_idx in matched_gt_indices:
                    continue

                iou = compute_iou(p_obj["box"], g_obj["box"])
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = g_idx

            if best_iou >= IOU_MATCH_THRESHOLD:
                gt_label = gt_objects[best_gt_idx]["label"]
                pred_label = p_obj["label"]

                if gt_label in CLASSES and pred_label in CLASSES:
                    conf_matrix[gt_label][pred_label] += 1

                matched_gt_indices.add(best_gt_idx)
            else:
                pred_label = p_obj["label"]
                if pred_label in CLASSES:
                    conf_matrix["Background"][pred_label] += 1

        for g_idx, g_obj in enumerate(gt_objects):
            if g_idx not in matched_gt_indices:
                gt_label = g_obj["label"]
                if gt_label in CLASSES:
                    conf_matrix[gt_label]["Background"] += 1

        del inputs, outputs, gen_tokens, pred_objects, gt_objects
        clear_vram()

        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1} / {len(dataset)} images...")

    # plot
    print("\nDone. Plotting confusion matrix...")

    matrix_data = []
    for true_cls in CLASSES:
        row = []
        for pred_cls in CLASSES:
            if true_cls == "Background" and pred_cls == "Background":
                row.append(0)
            else:
                row.append(conf_matrix[true_cls][pred_cls])
        matrix_data.append(row)

    df_cm = pd.DataFrame(matrix_data, index=CLASSES, columns=CLASSES)

    plt.figure(figsize=(8, 6))
    sns.heatmap(df_cm, annot=True, fmt='g', cmap='Blues', cbar=True, annot_kws={"size": 14})

    plt.title('Object Detection Confusion Matrix (IoU > 0.50)', fontsize=16)
    plt.ylabel('Ground Truth', fontsize=14)
    plt.xlabel('Prediction', fontsize=14)

    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

generate_confusion_matrix(test_dataset)

## 10. Deploy to Hugging Face Hub

In [ ]:
from google.colab import userdata
import gc
import torch

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

hf_write_token = userdata.get('write_hf')
repo_id = "Snadderr/paligemma-10b-896-military"

print(f"Uploading fine-tuned model to {repo_id}...")

model.push_to_hub(repo_id, token=hf_write_token)
processor.push_to_hub(repo_id, token=hf_write_token)

print("Done. Model uploaded to Hugging Face Hub.")

clear_vram()